In [1]:
import os
import pandas as pd
import boto3
import math
from sagemaker import get_execution_role
from pprint import pprint
import time

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.0' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Functions

In [2]:
def get_specs(str_instance):
    if str_instance == 'm5.large':
        int_vcpu = 2
        int_memory_gb = 8
    elif str_instance == 'm5.xlarge':
        int_vcpu = 4
        int_memory_gb = 16
    elif str_instance == 'm5.2xlarge':
        int_vcpu = 8
        int_memory_gb = 32
    elif str_instance == 'm5.4xlarge':
        int_vcpu = 16
        int_memory_gb = 64
    elif str_instance == 'm5.8xlarge':
        int_vcpu = 32
        int_memory_gb = 128
    elif str_instance == 'm5.12xlarge':
        int_vcpu = 48
        int_memory_gb = 192
    int_memory_mebibytes = math.ceil(int_memory_gb * 953.674)
    dict_output = {
        'int_vcpu': int_vcpu,
        'int_memory_gb': int_memory_gb,
        'int_memory_mebibytes': int_memory_mebibytes,
    }
    return dict_output

### Constants

In [3]:
str_project = os.getcwd().split('/')[4].replace('_','-')
print(f'Project: {str_project}')
str_image_name = 'genxii-pd-monitoring'
int_iteration = 1
str_instance = 'm5.large'
dict_specs = get_specs(str_instance=str_instance)
int_vcpu = dict_specs['int_vcpu']
int_memory_gb = dict_specs['int_memory_gb']
int_memory_mebibytes = dict_specs['int_memory_mebibytes']
for key, val in dict_specs.items():
    print(f'{key}: {val}')

Project: 20231010-gen-xii
int_vcpu: 2
int_memory_gb: 8
int_memory_mebibytes: 7630


### Create Hyperparameters Data Frame

In [4]:
# make a dictionary of hyperparameters, save as df to s3, so I can re-convert it to dict in the images
dict_hyperparameters = {
    # data sets
    'STR_FILENAME_TRAIN': 'df_train_noleaks_pre.gzip',
    'STR_FILENAME_VALID': 'df_valid_noleaks_pre.gzip', # always use the full data set for the valid model
    # ITERATIONS - define once for consistency
    'INT_N_ITERATIONS': 1000,
    # proportion of iterations used for early stopping
    'PROP_EARLY_STOPPING': 0.10,
    # eval metric
    'STR_EVAL_METRIC': 'AUC',
    # tuning jobs
    'INT_N_TUNING_JOBS': 100,
    # variant
    'STR_VARIANT': 'noPTImodel7',
}

# make df
df = pd.DataFrame(dict_hyperparameters.items(), columns=['keys','values'])

# save
str_filename = 'df_hyperparameters.csv'
str_uri = f's3://{str_project}/11_monitoring/input/{str_filename}'
df.to_csv(str_uri, index=False)

# show
df

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/fsspec/registry.py:272: UserWarning: Your installed version of s3fs is very old and known to cause
severe performance issues, see also https://github.com/dask/dask/issues/10276

To fix, you should specify a lower version bound on s3fs, or
update the current installation.

  warnings.warn(s3_msg)


,keys,values
0,STR_FILENAME_TRAIN,df_train_noleaks_pre.gzip
1,STR_FILENAME_VALID,df_valid_noleaks_pre.gzip
2,INT_N_ITERATIONS,1000
3,PROP_EARLY_STOPPING,0.1
4,STR_EVAL_METRIC,AUC
5,INT_N_TUNING_JOBS,100
6,STR_VARIANT,noPTImodel7


### Get the number of jobs in the array

In [5]:
str_filename = 'df_targets.csv'
str_uri = f's3://{str_project}/09_early_indicators/input/{str_filename}'
df = pd.read_csv(str_uri)
# get n rows
int_n_jobs = df.shape[0]
print(f'There will be {int_n_jobs} jobs in the array')

There will be 141 jobs in the array


### Create compute environment

In [6]:
# initialize class
cls_client = boto3.client('batch')

In [7]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [8]:
# create compute environment
while True:
    try:
        str_compute_env_name = f'env-{str_image_name}-{int_iteration}'
        dict_response = cls_client.create_compute_environment(
            computeEnvironmentName=str_compute_env_name,
            type= 'Managed', 
            state= 'ENABLED',
            serviceRole = str_role,
            computeResources={
                #'type': 'SPOT',
                'type': 'EC2',
                'minvCpus': 0,
                'maxvCpus': 256, 
                'desiredvCpus': int_vcpu,
                'instanceTypes': [
                    str_instance,
                ], 
                'subnets': ['subnet-044e573651bb251a7'], 
                'securityGroupIds': ['sg-03904237048cdc335'], 
                'instanceRole': 'ecsInstanceRole',
                #'spotIamFleetRole': 'AmazonEC2SpotFleetTaggingRole',
            },
        )
        pprint(dict_response)
        break
    except:
        int_iteration += 1

{'ResponseMetadata': {'HTTPHeaders': {'access-control-allow-origin': '*',
                                      'access-control-expose-headers': 'X-amzn-errortype,X-amzn-requestid,X-amzn-errormessage,X-amzn-trace-id,X-amz-apigw-id,date',
                                      'connection': 'keep-alive',
                                      'content-length': '165',
                                      'content-type': 'application/json',
                                      'date': 'Fri, 23 Feb 2024 19:00:43 GMT',
                                      'x-amz-apigw-id': 'Tmm2YFw_vHcEKfA=',
                                      'x-amzn-requestid': 'ba26636d-0816-49a2-8a8e-ddbf879e7467',
                                      'x-amzn-trace-id': 'Root=1-65d8eb5b-43392d9d4f05f8ca26434e3b'},
                      'HTTPStatusCode': 200,
                      'RequestId': 'ba26636d-0816-49a2-8a8e-ddbf879e7467',
                      'RetryAttempts': 0},
 'computeEnvironmentArn': 'arn:aws:batch:

### Create Job Queue

In [9]:
# create job queue (this is where AWS will store your jobs until an EC2 Instance is available to run them)
while True:
    try:
        str_job_queue_name = f'queue-{str_image_name}-{int_iteration}'
        dict_response = cls_client.create_job_queue(
            jobQueueName=str_job_queue_name,
            state='ENABLED',
            priority=1,
            computeEnvironmentOrder=[
                {
                    'order': 1,
                    'computeEnvironment': str_compute_env_name,
                },
            ]
        )
        # get arn
        str_job_queue_arn = dict_response['jobQueueArn']
        pprint(dict_response)
        break
    except:
        time.sleep(1)

{'ResponseMetadata': {'HTTPHeaders': {'access-control-allow-origin': '*',
                                      'access-control-expose-headers': 'X-amzn-errortype,X-amzn-requestid,X-amzn-errormessage,X-amzn-trace-id,X-amz-apigw-id,date',
                                      'connection': 'keep-alive',
                                      'content-length': '139',
                                      'content-type': 'application/json',
                                      'date': 'Fri, 23 Feb 2024 19:00:51 GMT',
                                      'x-amz-apigw-id': 'Tmm3kGpyvHcEP4w=',
                                      'x-amzn-requestid': '32d60e78-94ec-4669-80b9-66d2c6955962',
                                      'x-amzn-trace-id': 'Root=1-65d8eb63-579361a417e7358000effbca'},
                      'HTTPStatusCode': 200,
                      'RequestId': '32d60e78-94ec-4669-80b9-66d2c6955962',
                      'RetryAttempts': 0},
 'jobQueueArn': 'arn:aws:batch:us-west-2:

### Register job definition

In [10]:
# job definition
while True:
    try:
        str_job_definition = f'job-def-{str_image_name}-{int_iteration}'
        dict_response = cls_client.register_job_definition(
            type='container',
            containerProperties={
                'image': f'836690756591.dkr.ecr.us-west-2.amazonaws.com/{str_image_name}:latest',
                'memory': int_memory_mebibytes,
                'vcpus': int_vcpu,
            },
            jobDefinitionName=str_job_definition,
        )
        # get arn
        str_job_def_arn = dict_response['jobDefinitionArn']
        pprint(dict_response)
        break
    except:
        time.sleep(1)

{'ResponseMetadata': {'HTTPHeaders': {'access-control-allow-origin': '*',
                                      'access-control-expose-headers': 'X-amzn-errortype,X-amzn-requestid,X-amzn-errormessage,X-amzn-trace-id,X-amz-apigw-id,date',
                                      'connection': 'keep-alive',
                                      'content-length': '173',
                                      'content-type': 'application/json',
                                      'date': 'Fri, 23 Feb 2024 19:00:51 GMT',
                                      'x-amz-apigw-id': 'Tmm3lGTQvHcEIiA=',
                                      'x-amzn-requestid': '91ca6aa2-1526-4c9f-85f4-199b9ca3c50c',
                                      'x-amzn-trace-id': 'Root=1-65d8eb63-6da50308548dec9b0f32d9f1'},
                      'HTTPStatusCode': 200,
                      'RequestId': '91ca6aa2-1526-4c9f-85f4-199b9ca3c50c',
                      'RetryAttempts': 0},
 'jobDefinitionArn': 'arn:aws:batch:us-we

### Submit job

In [11]:
# submit a job
while True:
    try:
        str_job_name = f'job-name-{str_image_name}-{int_iteration}'
        response = cls_client.submit_job(
            jobDefinition=str_job_definition,
            jobQueue=str_job_queue_name,
            jobName=str_job_name,
            arrayProperties={
                'size': int_n_jobs,
            },
        )
        pprint(response)
        break
    except:
        time.sleep(1)

{'ResponseMetadata': {'HTTPHeaders': {'access-control-allow-origin': '*',
                                      'access-control-expose-headers': 'X-amzn-errortype,X-amzn-requestid,X-amzn-errormessage,X-amzn-trace-id,X-amz-apigw-id,date',
                                      'connection': 'keep-alive',
                                      'content-length': '181',
                                      'content-type': 'application/json',
                                      'date': 'Fri, 23 Feb 2024 19:00:53 GMT',
                                      'x-amz-apigw-id': 'Tmm37H12vHcEtOQ=',
                                      'x-amzn-requestid': 'af040eec-5dab-4e45-8049-008798a3e61b',
                                      'x-amzn-trace-id': 'Root=1-65d8eb65-2f48c3d95acb70ac51951897'},
                      'HTTPStatusCode': 200,
                      'RequestId': 'af040eec-5dab-4e45-8049-008798a3e61b',
                      'RetryAttempts': 0},
 'jobArn': 'arn:aws:batch:us-west-2:83669

### Show arns

In [12]:
print(f'Job Queue ARN: {str_job_queue_arn}')
print(f'Job Definition ARN: {str_job_def_arn}')

Job Queue ARN: arn:aws:batch:us-west-2:836690756591:job-queue/queue-genxii-pd-monitoring-6
Job Definition ARN: arn:aws:batch:us-west-2:836690756591:job-definition/job-def-genxii-pd-monitoring-6:2
